# IT9203 - Data Analysis with Hive and AI: Collaborative Filtering (ALS)

**Project:** Top-Rated Movies per Genre Tracker  
**Platform:** Azure Databricks  
**Notebook:** 05 - Data Analysis and AI (20 marks)  
**Tools:** Apache Hive (Spark SQL analytics) + Spark MLlib ALS (recommendation model)

## Overview

This notebook is divided into two parts:

### Part A - Apache Hive SQL Analysis
Run analytical queries against the Hive tables created in Notebook 04:
- Genre-level overview (rating volume, unique movies, avg rating)
- Top-10 movies per genre
- Yearly trend analysis
- Rating distribution

### Part B - AI: Collaborative Filtering with Spark MLlib ALS
Train an Alternating Least Squares (ALS) matrix factorisation model to generate personalised movie recommendations:
- MovieLens userId/movieId are already integers (no hashing needed)
- Train/test split and model training
- RMSE and MAE evaluation
- Generate top-10 recommendations per user
- Persist recommendations to DBFS and Hive

---
## PART A: Apache Hive SQL Analysis

### A1. Use Existing Spark Session

Databricks provides a pre-configured notebook-scoped Spark session with Hive support enabled. We simply reference the Hive database created in Notebook 04.

In [0]:
from pyspark.sql.functions import col

spark.sql("USE movielens")

print("Available Hive tables:")
spark.sql("SHOW TABLES IN movielens").show()

Available Hive tables:
+---------+---------------+-----------+
| database|      tableName|isTemporary|
+---------+---------------+-----------+
|movielens|ratings_cleaned|      false|
|movielens|     top_movies|      false|
+---------+---------------+-----------+



### A2. Genre Overview

In [0]:
spark.sql("""
    SELECT 
        genre,
        COUNT(*) AS total_ratings,
        COUNT(DISTINCT movieId) AS unique_movies,
        ROUND(AVG(rating), 2) AS avg_rating,
        ROUND(STD(rating), 3) AS rating_std
    FROM movielens.ratings_cleaned
    GROUP BY genre
    ORDER BY total_ratings DESC
""").show(20)

+------------------+-------------+-------------+----------+----------+
|             genre|total_ratings|unique_movies|avg_rating|rating_std|
+------------------+-------------+-------------+----------+----------+
|             Drama|     10962833|        24465|      3.68|     0.998|
|            Comedy|      8926230|        16051|      3.42|     1.083|
|            Action|      7446918|         6913|      3.47|     1.071|
|          Thriller|      6763272|         8330|      3.52|     1.038|
|         Adventure|      5832424|         3868|      3.52|     1.068|
|           Romance|      4497291|         7305|      3.54|     1.048|
|            Sci-Fi|      4325740|         3502|      3.48|     1.085|
|             Crime|      4190259|         5024|      3.69|      1.01|
|           Fantasy|      2831585|         2667|      3.51|     1.089|
|          Children|      2124258|         2866|      3.43|     1.098|
|           Mystery|      2010995|         2782|      3.67|     1.007|
|     

### A3. Top-10 Rated Movies Overall

In [0]:
spark.sql("""
    SELECT 
        tm.genre,
        tm.rank,
        tm.title,
        tm.rating_count,
        ROUND(tm.avg_rating, 2) AS avg_rating,
        ROUND(tm.popularity_score, 3) AS score
    FROM movielens.top_movies tm
    ORDER BY tm.popularity_score DESC
    LIMIT 10
""").show(truncate=50)

+--------+----+--------------------------------+------------+----------+------+
|   genre|rank|                           title|rating_count|avg_rating| score|
+--------+----+--------------------------------+------------+----------+------+
|   Crime|   1|Shawshank Redemption, The (1994)|       81482|      4.41|49.909|
|   Drama|   1|Shawshank Redemption, The (1994)|       81482|      4.41|49.909|
|  Comedy|   1|             Pulp Fiction (1994)|       79672|      4.19|47.275|
|   Drama|   2|             Pulp Fiction (1994)|       79672|      4.19|47.275|
|Thriller|   1|             Pulp Fiction (1994)|       79672|      4.19|47.275|
|   Crime|   2|             Pulp Fiction (1994)|       79672|      4.19|47.275|
|   Crime|   3|           Godfather, The (1972)|       52498|      4.32|46.999|
|   Drama|   3|           Godfather, The (1972)|       52498|      4.32|46.999|
|   Crime|   4|      Usual Suspects, The (1995)|       55366|      4.28|46.793|
| Mystery|   1|      Usual Suspects, The

### A4. Yearly Trend Analysis

In [0]:
spark.sql("""
    SELECT 
        YEAR(review_date) AS year,
        genre,
        COUNT(*) AS rating_count,
        ROUND(AVG(rating), 2) AS avg_rating
    FROM movielens.ratings_cleaned
    WHERE YEAR(review_date) >= 2005
    GROUP BY YEAR(review_date), genre
    ORDER BY year DESC, rating_count DESC
""").show(20)

+----+------------------+------------+----------+
|year|             genre|rating_count|avg_rating|
+----+------------------+------------+----------+
|2019|             Drama|      499516|       3.7|
|2019|            Action|      399546|      3.54|
|2019|            Comedy|      368382|      3.47|
|2019|          Thriller|      329395|      3.58|
|2019|         Adventure|      315235|      3.57|
|2019|            Sci-Fi|      261729|      3.57|
|2019|             Crime|      200572|      3.73|
|2019|           Romance|      161827|      3.55|
|2019|           Fantasy|      153567|      3.54|
|2019|         Animation|      107622|      3.62|
|2019|            Horror|      106956|      3.37|
|2019|           Mystery|      104434|       3.7|
|2019|          Children|      101212|      3.49|
|2019|              IMAX|       91532|      3.56|
|2019|               War|       51024|      3.77|
|2019|           Musical|       28768|      3.53|
|2019|       Documentary|       21730|      3.64|


### A5. Rating Distribution

In [0]:
spark.sql("""
    SELECT 
        rating,
        COUNT(*) AS rating_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage
    FROM movielens.ratings_cleaned
    GROUP BY rating
    ORDER BY rating
""").show()

+------+------------+----------+
|rating|rating_count|percentage|
+------+------------+----------+
|   0.5|     1055872|      1.56|
|   1.0|     2017049|      2.97|
|   1.5|     1094170|      1.61|
|   2.0|     4357049|      6.43|
|   2.5|     3473184|      5.12|
|   3.0|    13133949|     19.37|
|   3.5|     8758722|     12.92|
|   4.0|    17934256|     26.45|
|   4.5|     6113701|      9.02|
|   5.0|     9871934|     14.56|
+------+------------+----------+



---
## PART B: AI - Collaborative Filtering with ALS

### Background: Alternating Least Squares (ALS)

ALS is a matrix factorisation algorithm used for collaborative filtering. Given a sparse user-item rating matrix **R**, ALS factorises it into two lower-rank matrices:

```
R = U x V^T

Where:
  U = User latent factors matrix  (n_users x rank)
  V = Item latent factors matrix  (n_items x rank)
  rank = number of latent dimensions (hyperparameter)
```

The algorithm alternates between fixing **U** and solving for **V**, then fixing **V** and solving for **U** - each step being a least-squares problem with a closed-form solution.

**Why ALS for MovieLens?**
- MovieLens is the standard benchmark dataset for collaborative filtering research
- ALS handles the sparse user-movie rating matrix efficiently
- Spark MLlib's ALS implementation distributes both matrices across the cluster
- MovieLens provides native integer IDs, simplifying the pipeline

### B1. Prepare Data for ALS

Unlike the Amazon dataset, MovieLens userId and movieId are already integers - no hashing required.

In [0]:
from pyspark.sql.functions import col
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

als_df = spark.sql("""
    SELECT 
        userId,
        movieId AS item_id,
        CAST(rating AS FLOAT) AS rating
    FROM movielens.ratings_cleaned
    WHERE rating IS NOT NULL
    GROUP BY userId, movieId, rating
""").dropDuplicates(["userId", "item_id"])

als_count = als_df.count()
n_users = als_df.select("userId").distinct().count()
n_items = als_df.select("item_id").distinct().count()

print(f"ALS training records: {als_count:,}")
print(f"Unique users: {n_users:,}")
print(f"Unique items: {n_items:,}")
print(f"Sparsity: {1 - als_count / (n_users * n_items):.6f}")
als_df.show(5)

ALS training records: 25,000,095
Unique users: 162,541
Unique items: 59,047
Sparsity: 0.997395
+------+-------+------+
|userId|item_id|rating|
+------+-------+------+
| 62670|   2959|   5.0|
| 62805|   1206|   3.0|
| 63597|  52458|   5.0|
| 63875|   1944|   4.0|
| 65088|   4979|   5.0|
+------+-------+------+
only showing top 5 rows


### B2. Train/Test Split

In [0]:
train_df, test_df = als_df.randomSplit([0.8, 0.2], seed=42)

print(f"Training set: {train_df.count():,} records")
print(f"Test set:     {test_df.count():,} records")

Training set: 19,999,969 records
Test set:     5,000,126 records


### B3. Train ALS Model

In [0]:
import time

als = ALS(
    maxIter=15,
    regParam=0.01,
    rank=20,
    userCol="userId",
    itemCol="item_id",
    ratingCol="rating",
    coldStartStrategy="drop",
    implicitPrefs=False,
    nonnegative=True
)

t0 = time.time()
model = als.fit(train_df)
elapsed = time.time() - t0

print(f"ALS model trained successfully")
print(f"  Rank:    {model.rank}")
print(f"  maxIter: 15")
print(f"  regParam: 0.01")
print(f"  Training time: {elapsed:.1f}s")

ALS model trained successfully
  Rank:    20
  maxIter: 15
  regParam: 0.01
  Training time: 402.1s


### B4. Evaluate Model

In [0]:
predictions = model.transform(test_df)

evaluator_rmse = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)
evaluator_mae = RegressionEvaluator(
    metricName="mae",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = evaluator_rmse.evaluate(predictions)
mae = evaluator_mae.evaluate(predictions)

print("Model Evaluation Results:")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAE:  {mae:.4f}")
print(f"")
print(f"  Interpretation: On a 0.5-5.0 scale, predictions are")
print(f"  off by ~{rmse:.2f} stars (RMSE) or ~{mae:.2f} stars (MAE) on average.")
print(f"  These are competitive results for MovieLens 25M.")

print("\nSample predictions vs. actuals:")
predictions.select("userId", "item_id", "rating", "prediction") \
    .show(10)

Model Evaluation Results:
  RMSE: 0.8013
  MAE:  0.6066

  Interpretation: On a 0.5-5.0 scale, predictions are
  off by ~0.80 stars (RMSE) or ~0.61 stars (MAE) on average.
  These are competitive results for MovieLens 25M.

Sample predictions vs. actuals:
+------+-------+------+----------+
|userId|item_id|rating|prediction|
+------+-------+------+----------+
|     2|   1674|   4.0|  3.571887|
|     2|   2918|   4.5| 3.2908478|
|     2|   3479|   2.0| 3.2359571|
|     2|   6333|   5.0| 3.7431214|
|     3|      1|   4.0| 3.6185243|
|     3|   1217|   5.0| 4.2894416|
|     3|   1356|   4.0| 3.2741358|
|     3|   3156|   4.5| 3.4892664|
|     3|   3980|   4.0| 3.6820562|
|     3|   4844|   4.0| 3.5841117|
+------+-------+------+----------+
only showing top 10 rows


### B5. Generate Top-10 Recommendations per User

In [0]:
user_recs = model.recommendForAllUsers(10)
print(f"Generated recommendations for {user_recs.count():,} users")
user_recs.show(5, truncate=False)

Generated recommendations for 162,541 users
+------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|userId|recommendations                                                                                                                                                                                              |
+------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|1     |[{110752, 11.342972}, {134037, 10.696488}, {179065, 9.662278}, {152986, 9.309888}, {199217, 9.095929}, {133199, 9.047901}, {180245, 8.612886}, {170315, 8.601375}, {116847, 8.500397}, {140840, 8.467589}]   |
|6     |[{138580, 10.4652605}, {110752, 9.914768}, {109669, 9.352053}, {117604, 9.349521}, {8230

### B6. Join Recommendations with Movie Titles

In [0]:
from pyspark.sql.functions import explode

movie_lookup = spark.sql("""
    SELECT DISTINCT
        movieId AS item_id,
        title,
        genres
    FROM movielens.ratings_cleaned
""")

sample_user = 1

sample_recs = user_recs.filter(col("userId") == sample_user) \
    .select(col("userId"), explode(col("recommendations")).alias("rec")) \
    .select(col("userId"), col("rec.item_id").alias("item_id"), col("rec.rating").alias("pred_rating")) \
    .join(movie_lookup, on="item_id", how="left") \
    .select("userId", "pred_rating", "title", "genres") \
    .orderBy(col("pred_rating").desc())

print(f"Top-10 movie recommendations for user {sample_user}:")
sample_recs.show(10, truncate=50)

Top-10 movie recommendations for user 1:
+------+-----------+--------------------------------------------------+------------------------------+
|userId|pred_rating|                                             title|                        genres|
+------+-----------+--------------------------------------------------+------------------------------+
|     1|  11.342972|                            Mondo Hollywood (1967)|                   Documentary|
|     1|  10.696488|                           War-Time Romance (1983)|            (no genres listed)|
|     1|   9.662278|                             Scarred Hearts (2016)|                         Drama|
|     1|   9.309888|                   Bose: The Forgotten Hero (2005)|                  Action|Drama|
|     1|   9.095929|                                     Mother (2017)|                Drama|Thriller|
|     1|   9.047901|                              Working Girls (1986)|                         Drama|
|     1|   8.612886|            

### B7. Save Recommendations to DBFS and Hive

In [0]:
OUTPUT_PATH = "/FileStore/movielens/output/"

user_recs.write \
    .mode("overwrite") \
    .parquet(f"{OUTPUT_PATH}als_recommendations/")

print("Recommendations saved to DBFS")

user_recs.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("movielens.als_recommendations")

print("Recommendations saved to Hive table: movielens.als_recommendations")

display(dbutils.fs.ls(OUTPUT_PATH))

Recommendations saved to DBFS
Recommendations saved to Hive table: movielens.als_recommendations


path,name,size,modificationTime
dbfs:/FileStore/movielens/output/als_recommendations/,als_recommendations/,0,1779546492000


## Key Findings
### Hive Analysis
| Finding | Detail |
|---|---|
| Dataset | 25M ratings across 59,048 movies in 20 genres |
| Average rating | 3.53 out of 5 (balanced, unlike Amazon's 4.14) |
| Rating spread | Full 0.5-5.0 scale used, with 4.0 being the most common |
| Most-rated genre | Drama (highest volume across all years) |
| Top movie | Shawshank Redemption dominates genre rankings |
### ALS Model Performance
| Metric | Value | Interpretation |
|---|---|---|
| RMSE | 0.8013 | Predictions within ~0.80 stars of actual |
| MAE | 0.6066 | Mean absolute error ~0.61 stars |
| Training records | ~20M | 80% of 25M interactions |
| Recommendations generated | 162,541 users x 10 items | Full user coverage |
### Business Insights
1. **MovieLens is a cleaner ALS benchmark** - native integer IDs eliminate hashing, and the lower sparsity compared to Amazon produces better RMSE.
2. **Genre diversity matters** - movies spanning multiple genres naturally dominate per-genre rankings because they accumulate ratings from all their genres.
3. **ALS cold-start limitation** - new users with no rating history cannot receive personalised recommendations; a fallback to the top-movies-per-genre list is recommended.
4. **Rating scale difference** - MovieLens uses 0.5 increments (10-point scale effectively), providing more granularity than Amazon's integer 1-5 scale.